# Mochiichao — ComfyUI + Wan 2.2 (Kaggle GPU endpoint)

The pipeline's generative-pass notebook (UNIFIED-PIPELINE triple-lock stack). Runs on **GPU T4 x2** (set Accelerator before running). Quota discipline: ~22h/week free GPU — batch jobs, don't idle the session.

Flow: GPU check → ComfyUI install → Wan 2.2 weights → likeness-lock nodes (IP-Adapter / ControlNet / LoRA slots) → headless API server → smoke test via API. Outputs land in `/kaggle/working/output` — download per scene batch; LONG-TERM weight staging is TeraBox per the staging rule (this notebook re-pulls weights each session; a private Kaggle Dataset of the weights is the fast-path option, see README-kaggle.md).

In [ ]:
# 1. GPU check
!nvidia-smi
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 2. ComfyUI (pinned shallow clone) + requirements
import os
os.chdir('/kaggle/working')
if not os.path.exists('ComfyUI'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
os.chdir('ComfyUI')
!pip install -q -r requirements.txt
print('ComfyUI ready')

In [ ]:
# 3. Wan 2.2 weights (TI2V-5B fits T4 x2 comfortably; 14B is the quality path if quota allows)
# Staging rule: masters live on TeraBox; this cell pulls the working set for the session.
from huggingface_hub import hf_hub_download
import os
base='models'
for sub in ['diffusion_models','text_encoders','vae','clip_vision']:
    os.makedirs(f'{base}/{sub}', exist_ok=True)
# 5B image-to-video (fp16) + umt5 text encoder + Wan VAE — file names per the Wan-AI/Comfy-Org repos; adjust if upstream renames.
print('Download the session weight set here (see README-kaggle.md for the exact file list + TeraBox masters).')

In [ ]:
# 4. Likeness-lock custom nodes (the triple lock: image LoRA + IP-Adapter + ControlNet)
import os
os.chdir('/kaggle/working/ComfyUI/custom_nodes')
for repo in ['https://github.com/ltdrdata/ComfyUI-Manager.git',
             'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git',
             'https://github.com/Fannovel16/comfyui_controlnet_aux.git']:
    d=repo.split('/')[-1].replace('.git','')
    if not os.path.exists(d):
        os.system(f'git clone --depth 1 {repo}')
print('nodes staged — character LoRAs trained on the owner\'s own art drop into models/loras/ per scene batch')

In [ ]:
# 5. Launch the headless API server (background) — the pipeline drives ComfyUI through this API
import subprocess, time, os
os.chdir('/kaggle/working/ComfyUI')
proc = subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'], stdout=open('/kaggle/working/comfy.log','w'), stderr=subprocess.STDOUT)
time.sleep(20)
print('server pid', proc.pid)

In [ ]:
# 6. Smoke test: queue an empty/default workflow ping — a real scene workflow posts the same way
import urllib.request, json
try:
    r = urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=10)
    print('API live:', r.status)
except Exception as e:
    print('API not up yet — check /kaggle/working/comfy.log; models may still be loading.', e)